<a href="https://colab.research.google.com/github/omarcordero1/-omarcordero1/blob/main/Enriquecimiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Vamos a armar un script para enriquecer datos periodisticos** 🙂

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Cargamos libreria**

In [ ]:
import subprocess, sys
print("📦 Instalando librerías...")
for paquete in ['requests', 'beautifulsoup4', 'lxml', 'language-tool-python',
                 'textblob', 'textstat', 'transformers', 'torch', 'pandas', 'openpyxl', 'nltk']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
print("✅ Listo\n")

import nltk, requests, pandas as pd, numpy as np, re, warnings
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
import textstat
from language_tool_python import LanguageTool
from transformers import pipeline
from datetime import datetime

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

📦 Instalando librerías...
✅ Listo



True

**Analizador**

In [ ]:
class AnalizadorContenido:
    def __init__(self):
        print("🚀 Inicializando...")
        self.pipeline_sent = pipeline("sentiment-analysis",
                                      model="nlptown/bert-base-multilingual-uncased-sentiment",
                                      device=-1)
        self.tool_lenguaje = LanguageTool('es')
        self.stopwords_es = set(stopwords.words('spanish'))
        print("✅ Listo\n")

    def scrapear(self, url):
        try:
            resp = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
            soup = BeautifulSoup(resp.content, 'lxml')
            titulo = soup.find('h1')
            titulo = titulo.get_text(strip=True) if titulo else "N/A"
            contenido_elem = soup.find('article') or soup.find('main') or soup.find('body')
            for tag in contenido_elem.find_all(['script', 'style']):
                tag.decompose()
            parrafos = contenido_elem.find_all('p')
            contenido = ' '.join([p.get_text(strip=True) for p in parrafos])
            contenido = re.sub(r'\s+', ' ', contenido).strip()
            return {'titulo': titulo, 'contenido': contenido, 'exito': True, 'error': None}
        except Exception as e:
            return {'titulo': None, 'contenido': None, 'exito': False, 'error': str(e)}

    def analizar_ortografia(self, texto):
        try:
            errores = self.tool_lenguaje.check(texto)
            total_palabras = len(texto.split())
            cantidad_errores = len(errores)
            score = max(0, 100 - (cantidad_errores / max(total_palabras, 1) * 100))
            return {
                'total_errores': cantidad_errores,
                'score_ortografia': round(score, 2),
                'densidad_errores': round(cantidad_errores / max(total_palabras, 1) * 100, 2)
            }
        except:
            return {'total_errores': 0, 'score_ortografia': 100, 'densidad_errores': 0}

    def analizar_sentimiento(self, texto):
        try:
            texto_lim = ' '.join(texto.split()[:512])
            resultado = self.pipeline_sent(texto_lim)[0]
            sent_map = {'POSITIVE': 'Positivo', 'NEGATIVE': 'Negativo', 'NEUTRAL': 'Neutral'}
            return {
                'sentimiento': sent_map.get(resultado['label'], resultado['label']),
                'score_sentimiento': round(resultado['score'], 3)
            }
        except:
            return {'sentimiento': 'Neutral', 'score_sentimiento': 0}

    def analizar_calidad(self, texto):
        try:
            oraciones = sent_tokenize(texto)
            palabras = texto.split()
            num_oraciones = max(len(oraciones), 1)
            num_palabras = len(palabras)

            flesch = textstat.flesch_reading_ease(texto)
            flesch_grade = textstat.flesch_kincaid_grade(texto)

            palabras_limpias = [p for p in palabras if p.lower() not in self.stopwords_es]
            palabras_unicas = len(set(palabras_limpias))
            densidad = (palabras_unicas / len(palabras_limpias) * 100) if palabras_limpias else 0

            score_calidad = (flesch / 100 * 70) + (densidad / 100 * 30)

            return {
                'num_palabras': num_palabras,
                'num_oraciones': num_oraciones,
                'promedio_palabras_oracion': round(num_palabras / num_oraciones, 2),
                'flesch_reading_ease': round(flesch, 2),
                'flesch_kincaid_grade': round(flesch_grade, 2),
                'score_calidad_redaccion': round(score_calidad, 2),
                'densidad_vocabulario': round(densidad, 2)
            }
        except:
            return {'num_palabras': 0, 'score_calidad_redaccion': 0}

    def analizar_estructura(self, texto, titulo=''):
        try:
            oraciones = sent_tokenize(texto)
            tiene_apertura = len(oraciones) > 0 and len(oraciones[0].split()) >= 5
            numeros = re.findall(r'\b\d+(?:\.\d+)?\b', texto)
            citas = re.findall(r'["\']([^"\']+)["\']', texto)
            verbos = ['dijo', 'afirmó', 'informó', 'declaró', 'comentó', 'expresó', 'señaló']
            menciones = sum(1 for verbo in verbos if verbo in texto.lower())

            score_struct = 0
            if tiene_apertura: score_struct += 20
            if len(numeros) > 0: score_struct += 20
            if len(citas) > 0: score_struct += 20
            if menciones > 0: score_struct += 20

            palabras = texto.lower().split()
            palabras_limpias = [p for p in palabras if p not in self.stopwords_es and len(p) > 3]
            diversidad = len(set(palabras_limpias)) / len(palabras_limpias) if palabras_limpias else 0

            if diversidad > 0.3: score_struct += 20

            return {
                'cantidad_datos_numericos': len(numeros),
                'cantidad_citas': len(citas),
                'menciones_fuentes': menciones,
                'score_estructura': score_struct
            }
        except:
            return {'score_estructura': 0}

    def procesar(self, url=None, titulo=None, contenido=None):
        resultado = {'timestamp': datetime.now().isoformat(), 'url': url}

        if url:
            print(f"📡 Scrapeando: {url}")
            scrape = self.scrapear(url)
            if not scrape['exito']:
                resultado['error'] = scrape['error']
                return resultado
            titulo = scrape['titulo']
            contenido = scrape['contenido']

        if not contenido:
            resultado['error'] = 'Sin contenido'
            return resultado

        resultado['titulo'] = titulo
        resultado['ortografia'] = self.analizar_ortografia(contenido)
        resultado['sentimiento'] = self.analizar_sentimiento(contenido)
        resultado['calidad'] = self.analizar_calidad(contenido)
        resultado['estructura'] = self.analizar_estructura(contenido, titulo)

        return resultado

**Configuración**

In [ ]:
RUTA_ENTRADA = '/content/articulos.xlsx'  # ← CAMBIAR SEGÚN TU RUTA
RUTA_SALIDA = '/content/articulos_ANALIZADO.xlsx'

print("📁 Rutas configuradas")
print(f"   Entrada: {RUTA_ENTRADA}")
print(f"   Salida: {RUTA_SALIDA}\n")

# 4. CARGAR DATOS
try:
    df = pd.read_excel(RUTA_ENTRADA)
    print(f"✅ Excel cargado: {df.shape[0]} filas\n")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Verifica que la ruta sea correcta")
    raise

📁 Rutas configuradas
   Entrada: /content/articulos.xlsx
   Salida: /content/articulos_ANALIZADO.xlsx

✅ Excel cargado: 5000 filas



**Analizador**

In [ ]:
print("🔄 ANALIZANDO ARTÍCULOS\n")
analizador = AnalizadorContenido()
resultados = []

for idx, row in df.iterrows():
    print(f"[{idx+1}/{len(df)}] ", end='')
    url = row.get('url') if 'url' in df.columns else None
    titulo = row.get('titulo') if 'titulo' in df.columns else None
    contenido = row.get('contenido') if 'contenido' in df.columns else None

    resultado = analizador.procesar(url=url, titulo=titulo, contenido=contenido)
    resultados.append(resultado)
    print("✅")

🔄 ANALIZANDO ARTÍCULOS

🚀 Inicializando...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu
INFO:language_tool_python.download_lt:Unzipping /tmp/tmpz5qveb8e.zip to /root/.cache/language_tool_python.
INFO:language_tool_python.download_lt:Downloaded https://internal1.languagetool.org/snapshots/LanguageTool-latest-snapshot.zip to /root/.cache/language_tool_python.


✅ Listo

[1/5000] 📡 Scrapeando: https://www.milenio.com/comunidad/pension-para-mujeres-y-hombres-30-64-anos-requisitos
✅
[2/5000] 📡 Scrapeando: https://www.milenio.com/comunidad/marcha-de-generacion-z-en-cdmx-hoy-15-noviembre-2025-en-vivo


Token indices sequence length is longer than the specified maximum sequence length for this model (675 > 512). Running this sequence through the model will result in indexing errors


Se han truncado las últimas 5000 líneas del flujo de salida.
[2501/5000] 📡 Scrapeando: https://www.milenio.com/espectaculos/famosos/poderoso-mensaje-fatima-bosch-exponer-violencia-miss-universo
✅
[2502/5000] 📡 Scrapeando: https://www.milenio.com/espectaculos/bad-bunny-noche-grammy-latinos-vegas
✅
[2503/5000] 📡 Scrapeando: https://www.milenio.com/espectaculos/famosos/disfraces-famosos-celebridades-halloween-2025
✅
[2504/5000] 📡 Scrapeando: https://www.milenio.com/cultura/concierto-juan-gabriel-170-mil-personas-zocalo
✅
[2505/5000] 📡 Scrapeando: https://www.milenio.com/espectaculos/famosos/luisito-comunica-reacciona-al-asesinato-de-carlos-manzo
✅
[2506/5000] 📡 Scrapeando: https://www.milenio.com/politica/sheinbaum-pide-universidades-austeridad-ampliar-matricula
✅
[2507/5000] 📡 Scrapeando: https://www.milenio.com/espectaculos/famosos/filtran-audio-nieto-maribel-guardia-hablando-mal-ella
✅
[2508/5000] 📡 Scrapeando: https://www.milenio.com/opinion/jesus-rangel/estira-afloja/gobernacion-t-me

**Datos enriquecidos**

In [ ]:
print("\n📊 CREANDO DATAFRAME ENRIQUECIDO\n")

datos = []
for r in resultados:
    fila = {
        'URL': r.get('url', ''),
        'Título': r.get('titulo', ''),
        'Timestamp': r.get('timestamp', ''),
        'Score_Ortografia': r.get('ortografia', {}).get('score_ortografia', 0),
        'Errores': r.get('ortografia', {}).get('total_errores', 0),
        'Sentimiento': r.get('sentimiento', {}).get('sentimiento', ''),
        'Score_Sentimiento': r.get('sentimiento', {}).get('score_sentimiento', 0),
        'Palabras': r.get('calidad', {}).get('num_palabras', 0),
        'Oraciones': r.get('calidad', {}).get('num_oraciones', 0),
        'Flesch_Ease': r.get('calidad', {}).get('flesch_reading_ease', 0),
        'Score_Calidad': r.get('calidad', {}).get('score_calidad_redaccion', 0),
        'Datos_Numericos': r.get('estructura', {}).get('cantidad_datos_numericos', 0),
        'Citas': r.get('estructura', {}).get('cantidad_citas', 0),
        'Fuentes': r.get('estructura', {}).get('menciones_fuentes', 0),
        'Score_Estructura': r.get('estructura', {}).get('score_estructura', 0),
        'Score_General': round(np.mean([
            r.get('ortografia', {}).get('score_ortografia', 0),
            r.get('calidad', {}).get('score_calidad_redaccion', 0),
            r.get('estructura', {}).get('score_estructura', 0)
        ]), 2)
    }
    datos.append(fila)

df_final = pd.DataFrame(datos)

print(df_final.to_string())

Se han truncado las últimas 5000 líneas del flujo de salida.
0                                                               https://www.milenio.com/comunidad/pension-para-mujeres-y-hombres-30-64-anos-requisitos                                                     Registro Pensión Hombres y Mujeres de 30 a 64 Años: estos son TODOS los requisitos que necesitas cubrir para recibir 3 mil 200 pesos  2025-11-17T16:47:50.433199             91.11       20      1 star              0.443         0          0            0              0                0      0        0                 0          30.37
1                                                       https://www.milenio.com/comunidad/marcha-de-generacion-z-en-cdmx-hoy-15-noviembre-2025-en-vivo                                                                                       Marcha Generación Z EN VIVO: Últimas noticias de la manifestación en CdMx HOY 15 de noviembre 2025  2025-11-17T16:48:05.418107             97.54       77     Neutral

**Guardar**

print("\n\n💾 GUARDANDO RESULTADOS\n")
df_final.to_excel(RUTA_SALIDA, index=False, sheet_name='Análisis')
print(f"✅ Guardado en: {RUTA_SALIDA}")

**Estadisticas**

In [ ]:
print("\n📈 ESTADÍSTICAS FINALES\n")
print(f"Score Ortografía Promedio: {df_final['Score_Ortografia'].mean():.1f}/100")
print(f"Score Calidad Promedio: {df_final['Score_Calidad'].mean():.1f}/100")
print(f"Score Estructura Promedio: {df_final['Score_Estructura'].mean():.1f}/100")
print(f"Score General Promedio: {df_final['Score_General'].mean():.1f}/100")
print(f"\nSentimientos: {df_final['Sentimiento'].value_counts().to_dict()}")

print("\n✅ ¡ANÁLISIS COMPLETADO!")


📈 ESTADÍSTICAS FINALES

Score Ortografía Promedio: 87.8/100
Score Calidad Promedio: 0.0/100
Score Estructura Promedio: 0.0/100
Score General Promedio: 29.3/100

Sentimientos: {'Neutral': 2257, '1 star': 1304, '4 stars': 569, '2 stars': 336, '': 304, '3 stars': 126, '5 stars': 104}

✅ ¡ANÁLISIS COMPLETADO!


**Descarga**

In [ ]:
print(df_final.to_string())

                                                                                                                                                   URL                                                                                                                                                                                   Título                   Timestamp  Score_Ortografia  Errores Sentimiento  Score_Sentimiento  Palabras  Oraciones  Flesch_Ease  Score_Calidad  Datos_Numericos  Citas  Fuentes  Score_Estructura  Score_General
0                                                               https://www.milenio.com/comunidad/pension-para-mujeres-y-hombres-30-64-anos-requisitos                                                     Registro Pensión Hombres y Mujeres de 30 a 64 Años: estos son TODOS los requisitos que necesitas cubrir para recibir 3 mil 200 pesos  2025-11-17T16:47:50.433199             91.11       20      1 star              0.443         0          0            0        

In [ ]:
df_final.to_excel("df_final.xlsx", index=False)

In [ ]:
from google.colab import files
files.download("df_final.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>